In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime
import os 
import requests
import time


symbol = "BTC-USDT"
timeframe = "1day"
market_type = "spot"
start_time = "2021-01-01 00:00:00"
end_time = datetime.utcnow().strftime("%Y-%m-%d %H:%M:%S")
# end_time = "2021-06-10 00:00:00"

skip_fetch = False
if not skip_fetch:
    print("Fetching data...")
    def fetch_kucoin_candles_chunk(symbol, market_type="spot", timeframe="1min", start_time=None, end_time=None):
        time.sleep(0.2)  # Rate limiting
        base_url = "https://api.kucoin.com" if market_type.lower() == "spot" else "https://api-futures.kucoin.com"
        url = base_url + "/api/v1/market/candles"
        params = {"type": timeframe, "symbol": symbol.upper()}
        if start_time:
            params["startAt"] = int(time.mktime(time.strptime(start_time, "%Y-%m-%d %H:%M:%S")))
        if end_time:
            params["endAt"] = int(time.mktime(time.strptime(end_time, "%Y-%m-%d %H:%M:%S")))
        try:
            resp = requests.get(url, params=params, timeout=10)
            resp.raise_for_status()
            data = resp.json()
            if data.get("code") == "200000":
                return data["data"]
            else:
                raise Exception(f"KuCoin API error: {data}")
        except requests.exceptions.RequestException as e:
            raise Exception(f"Request failed: {str(e)}")

    def fetch_all_kucoin_candles(symbol, market_type="spot", timeframe="1min", start_time=None, end_time=None):
        chunks = []
        current_end = end_time
        start_timestamp = int(time.mktime(time.strptime(start_time, "%Y-%m-%d %H:%M:%S")))
        
        while True:
            try:
                chunk = fetch_kucoin_candles_chunk(symbol, market_type, timeframe, start_time, current_end)
                if not chunk:
                    print("No more data available")
                    break
                earliest_ts = int(chunk[-1][0])
                print(f"Fetched {len(chunk)} candles from {datetime.fromtimestamp(earliest_ts)}")
                chunks.extend(chunk)
                if earliest_ts <= start_timestamp:
                    print("Reached start time")
                    break
                current_end = time.strftime("%Y-%m-%d %H:%M:%S", time.gmtime(earliest_ts - 60))
            except Exception as e:
                print(f"Error occurred: {str(e)}")
                break
        
        if not chunks:
            return []
        
        chunks.sort(key=lambda x: x[0])  # Sort by timestamp
        result = [candle for candle in chunks if start_timestamp <= int(candle[0]) <= int(time.mktime(time.strptime(end_time, "%Y-%m-%d %H:%M:%S")))]
        print(f"Total candles fetched: {len(result)}")
        return result


    data = fetch_all_kucoin_candles(symbol, market_type, timeframe, start_time, end_time)
    print(f"Final data length: {len(data)}")
    data
else:
    print("Skipping fetch...")

In [11]:
"""Backtesting.py experiment script: optimize and analyze close-to-win excursion metrics."""

from backtesting import Backtest, Strategy
from backtesting.lib import crossover
from backtesting.test import GOOG
import pandas as pd


class SmaCross(Strategy):
    fast = 10
    slow = 30

    def init(self):
        close = self.data.Close
        self.sma_fast = self.I(lambda: pd.Series(close).rolling(self.fast).mean().values)
        self.sma_slow = self.I(lambda: pd.Series(close).rolling(self.slow).mean().values)

    def next(self):
        if crossover(self.sma_fast, self.sma_slow):
            self.buy()
        elif crossover(self.sma_slow, self.sma_fast):
            self.position.close()


def run_baseline():
    bt = Backtest(
        GOOG,
        SmaCross,
        cash=10_000,
        commission=0.002,
        finalize_trades=True,
    )
    stats = bt.run()
    print("\n=== Baseline stats ===")
    print(stats)
    return bt, stats


def add_excursions(stats):
    trades = stats['_trades'].copy()
    trades['MAE'] = trades['EntryPrice'] - trades['Entry_\u03bb']
    trades['MFE'] = trades['Exit_\u03bb'] - trades['EntryPrice']
    trades['ratio_SL'] = (trades['MAE'] / (trades['EntryPrice'] - trades['SL'])).abs().clip(0, 1)
    trades['ratio_TP'] = (trades['MFE'] / (trades['TP'] - trades['EntryPrice'])).abs().clip(0, 1)
    trades[['ratio_SL', 'ratio_TP']] = trades[['ratio_SL', 'ratio_TP']].fillna(0)
    return trades


def optimize_strategy(bt):
    print("\n=== Optimizing strategy parameters ===")
    opt, heatmap = bt.optimize(
        fast=range(5, 21),
        slow=range(20, 61, 2),
        maximize='SQN',
        method='grid',
        constraint=lambda p: p.fast < p.slow,
        return_heatmap=True,
        random_state=42,
    )

    # Best result
    best = opt["_strategy"]
    print(f"\nBest setup : fast={best.fast}, slow={best.slow}")
    print(f"Return [%] : {opt['Return [%]']:.2f}")
    print(f"SQN        : {opt['SQN']:.4f}")
    print(f"Max DD [%] : {opt['Max. Drawdown [%]']:.2f}")

    # All combinations, sorted by SQN descending
    all_results = (
        heatmap
        .reset_index()
        .rename(columns={0: 'SQN'})
        .sort_values('SQN', ascending=False)
    )
    print(f"\n=== All {len(all_results)} tested setups (sorted by SQN) ===")
    print(all_results.to_string(index=False))

    return opt, heatmap


def main():
    bt, stats = run_baseline()
    trades = add_excursions(stats)
    print("\n=== Top 8 trades by ratio_TP (closest to target) ===")
    print(trades.sort_values('ratio_TP', ascending=False)
                 .head(8)
                 .loc[:, ['EntryBar', 'ExitBar', 'EntryPrice', 'ExitPrice', 'MAE', 'MFE', 'ratio_TP', 'PnL']]
                 .to_string(index=False))

    # optimization run to find better config
    optimize_strategy(bt)

if __name__ == '__main__':
    main()



=== Baseline stats ===
Start                     2004-08-19 00:00:00
End                       2013-03-01 00:00:00
Duration                   3116 days 00:00:00
Exposure Time [%]                    57.58845
Equity Final [$]                  49843.29894
Equity Peak [$]                   53976.81972
Commissions [$]                    4318.39106
Return [%]                          398.43299
Buy & Hold Return [%]               522.06019
Return (Ann.) [%]                    20.73749
Volatility (Ann.) [%]                27.19898
CAGR [%]                             13.87214
Sharpe Ratio                          0.76244
Sortino Ratio                         1.48355
Calmar Ratio                          0.66493
Alpha [%]                            180.8473
Beta                                  0.41678
Max. Drawdown [%]                   -31.18765
Avg. Drawdown [%]                    -4.61149
Max. Drawdown Duration      584 days 00:00:00
Avg. Drawdown Duration       48 days 00:00:00
# Trades  

/Users/andre/Documents/Python_local/pine_script/.venv/lib/python3.14/site-packages/backtesting/backtesting.py:1624: UserWarning: Searching for best of 335 configurations.
  output = _optimize_grid()
/Users/andre/Documents/Python_local/pine_script/.venv/lib/python3.14/site-packages/backtesting/backtesting.py:1624: RuntimeWarning: If you want to use multi-process optimization with `multiprocessing.get_start_method() == 'spawn'` (e.g. on Windows),set `backtesting.Pool = multiprocessing.Pool` (or of the desired context) and hide `bt.optimize()` call behind a `if __name__ == '__main__'` guard. Currently using thread-based paralellism, which might be slightly slower for non-numpy / non-GIL-releasing code. See https://github.com/kernc/backtesting.py/issues/1256
  output = _optimize_grid()



Best setup : fast=7, slow=40
Return [%] : 454.62
SQN        : 2.5242
Max DD [%] : -24.26

=== All 335 tested setups (sorted by SQN) ===
 fast  slow      SQN
    7    40 2.524180
    5    26 2.482891
    9    40 2.459439
    8    38 2.445731
    9    36 2.438204
    8    40 2.437118
    5    34 2.426228
    5    22 2.423406
   10    34 2.419471
    7    34 2.417826
   10    20 2.403223
    5    28 2.392329
    8    36 2.381251
    7    38 2.376381
    5    20 2.347778
   13    34 2.342804
    5    24 2.332527
    8    42 2.313748
    6    22 2.299920
   11    34 2.294886
   10    38 2.287588
    6    24 2.282944
    6    44 2.282159
   10    40 2.271649
    9    38 2.264609
    8    34 2.263340
    7    22 2.248632
   13    32 2.245355
   12    20 2.243936
    9    42 2.228337
   11    32 2.223379
   12    42 2.213998
    8    20 2.207312
    9    22 2.204464
    7    36 2.196512
   10    42 2.192553
    6    34 2.185925
    8    30 2.184750
    9    20 2.181882
    7    20 2.179006
  

In [10]:
opt = optimize_strategy(bt)
best = opt["_strategy"]
print("Best strategy object:", best)
print("fast =", best.fast)
print("slow =", best.slow)


=== Optimizing strategy parameters ===


/Users/andre/Documents/Python_local/pine_script/.venv/lib/python3.14/site-packages/backtesting/backtesting.py:1624: UserWarning: Searching for best of 335 configurations.
  output = _optimize_grid()
/Users/andre/Documents/Python_local/pine_script/.venv/lib/python3.14/site-packages/backtesting/backtesting.py:1624: RuntimeWarning: If you want to use multi-process optimization with `multiprocessing.get_start_method() == 'spawn'` (e.g. on Windows),set `backtesting.Pool = multiprocessing.Pool` (or of the desired context) and hide `bt.optimize()` call behind a `if __name__ == '__main__'` guard. Currently using thread-based paralellism, which might be slightly slower for non-numpy / non-GIL-releasing code. See https://github.com/kernc/backtesting.py/issues/1256
  output = _optimize_grid()


Best parameters: Start                     2004-08-19 00:00:00
End                       2013-03-01 00:00:00
Duration                   3116 days 00:00:00
Exposure Time [%]                     57.5419
Equity Final [$]                  55461.50308
Equity Peak [$]                   56196.04948
Commissions [$]                    3796.53692
Return [%]                          454.61503
Buy & Hold Return [%]               467.73944
Return (Ann.) [%]                    22.25987
Volatility (Ann.) [%]                28.31178
CAGR [%]                             14.85999
Sharpe Ratio                          0.78624
Sortino Ratio                          1.5755
Calmar Ratio                          0.91747
Alpha [%]                           250.68258
Beta                                    0.436
Max. Drawdown [%]                   -24.26211
Avg. Drawdown [%]                    -4.82959
Max. Drawdown Duration      433 days 00:00:00
Avg. Drawdown Duration       47 days 00:00:00
# Trades         

In [6]:
bt.plot()

GridPlot(id='p1664', ...)

In [7]:
print(stats["_trades"])

    Size  EntryBar  ExitBar  EntryPrice  ExitPrice    SL    TP          PnL  \
0     53        86      113      186.31     193.69  None  None    350.86000   
1     52       119      128      196.96     196.50  None  None    -64.83984   
2     52       160      243      193.69     297.50  None  None   5347.03624   
3     52       267      294      297.28     304.96  None  None    336.72704   
4     46       299      365      345.78     430.57  None  None   3828.91580   
5     50       408      434      389.53     408.31  None  None    859.21600   
6     53       459      486      386.62     385.02  None  None   -166.59384   
7     54       518      585      376.72     484.69  None  None   5737.34772   
8     52       604      623      501.99     471.65  None  None  -1678.93856   
9     53       657      686      462.10     461.83  None  None   -112.24658   
10    50       698      740      484.50     512.92  None  None   1321.25800   
11    49       765      821      515.02     643.77  